In [ ]:
# =====================================================================
# PHYSICAL AI - AUTONOMOUS CAR NAVIGATION DEMO
# GOOGLE COLAB - SINGLE CELL
#
# Real Highway Video + YOLO + Lane Detection + TTC + Gemini
# + Safety Controller + Steering + Throttle + Brake
#
# EDUCATIONAL PHYSICAL AI DEMONSTRATION ONLY
# NOT FOR REAL VEHICLE CONTROL
# =====================================================================


# =====================================================================
# 1. INSTALL LIBRARIES
# =====================================================================

!pip install -q ultralytics google-genai opencv-python


# =====================================================================
# 2. IMPORT LIBRARIES
# =====================================================================

import cv2
import numpy as np
import math
import os
import json
import time

from ultralytics import YOLO

from google import genai
from google.colab import userdata, files

from IPython.display import HTML, display
from base64 import b64encode



In [ ]:
# =====================================================================
# STEP 2
# GEMINI SETUP
#
# In Colab:
#
# Left sidebar
#      ↓
# Secrets
#      ↓
# Add:
#
# GEMINI_API_KEY
#
# =====================================================================

USE_GEMINI = True
import os
from google import genai
from google.colab import userdata

#API_KEY = put the new api key here created latest
api_key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)
os.environ["GOOGLE_API_KEY"] = api_key
#import os
#from getpass import getpass

#os.environ["GOOGLE_API_KEY"] = getpass("Enter Gemini API key: ")

In [ ]:


# =====================================================================
# 4. UPLOAD A REAL HIGHWAY / DASHCAM VIDEO
# =====================================================================

print()
print("======================================================")
print("UPLOAD A HIGHWAY / DASHCAM VIDEO")
print("Recommended duration: 15-30 seconds")
print("======================================================")
print()

uploaded = files.upload()

input_video = list(uploaded.keys())[0]

print()
print("Input video:", input_video)



In [ ]:
# =====================================================================
# 5. LOAD YOLO OBJECT DETECTOR
# =====================================================================

print()
print("Loading YOLO...")
model = YOLO("yolo11n.pt")
print("YOLO loaded successfully.")

# =====================================================================
# 6. OPEN INPUT VIDEO
# =====================================================================

cap = cv2.VideoCapture(input_video)
if not cap.isOpened():
    raise Exception(
        "Could not open the uploaded highway video."
    )
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 25
width = int(
    cap.get(cv2.CAP_PROP_FRAME_WIDTH)
)
height = int(
    cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
)
total_frames = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)

# Limit demonstration to approximately 20 seconds
MAX_SECONDS = 20
max_frames = min(
    total_frames,
    int(fps * MAX_SECONDS)
)

print()
print("Video information")
print("------------------------")
print("Width        :", width)
print("Height       :", height)
print("FPS          :", round(fps, 2))
print("Frames used  :", max_frames)

# =====================================================================
# 7. CREATE OUTPUT VIDEO
# =====================================================================

raw_output = "autonomous_highway_raw.mp4"

final_output = "autonomous_highway_demo.mp4"


writer = cv2.VideoWriter(

    raw_output,

    cv2.VideoWriter_fourcc(*"mp4v"),

    fps,

    (width, height)

)


In [ ]:
# =====================================================================
# 8. YOLO CLASSES
# =====================================================================
#
# COCO:
#
# 0 = person
# 1 = bicycle
# 2 = car
# 3 = motorcycle
# 5 = bus
# 7 = truck
#
# =====================================================================

ROAD_CLASSES = [0, 1, 2, 3, 5, 7]


CLASS_NAMES = {

    0: "Pedestrian",

    1: "Bicycle",

    2: "Car",

    3: "Motorcycle",

    5: "Bus",

    7: "Truck"

}


CLASS_COLORS = {

    "Pedestrian": (0,255,255),

    "Bicycle": (255,255,0),

    "Car": (0,255,0),

    "Motorcycle": (0,165,255),

    "Bus": (255,0,255),

    "Truck": (0,0,255)

}


# =====================================================================
# 9. AUTONOMOUS VEHICLE STATE
# =====================================================================

ego_speed = 85.0

target_speed = 90.0


steering_angle = 0.0

throttle = 30.0

brake = 0.0


gemini_action = "KEEP_LANE"

gemini_reason = "Initial cruise"

final_action = "KEEP_LANE"

safety_status = "APPROVED"


estimated_distance = float("inf")

current_ttc = float("inf")


lane_status = "CENTERED"


# Previous bounding-box information
# Used for approximate relative-motion estimation

previous_box_heights = {}


# =====================================================================
# 10. LANE DETECTION
# =====================================================================

def detect_lanes(frame):

    """
    OpenCV lane detection.
    Compatible with different HoughLinesP output shapes.
    """

    h, w = frame.shape[:2]

    # ---------------------------------------------------------
    # 1. Convert to grayscale
    # ---------------------------------------------------------

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    # ---------------------------------------------------------
    # 2. Gaussian blur
    # ---------------------------------------------------------

    blur = cv2.GaussianBlur(
        gray,
        (5, 5),
        0
    )

    # ---------------------------------------------------------
    # 3. Edge detection
    # ---------------------------------------------------------

    edges = cv2.Canny(
        blur,
        60,
        160
    )

    # ---------------------------------------------------------
    # 4. Road region of interest
    # ---------------------------------------------------------

    mask = np.zeros_like(edges)

    polygon = np.array(
        [[
            (int(w * 0.10), h),
            (int(w * 0.42), int(h * 0.58)),
            (int(w * 0.58), int(h * 0.58)),
            (int(w * 0.92), h)
        ]],
        dtype=np.int32
    )

    cv2.fillPoly(
        mask,
        polygon,
        255
    )

    road_edges = cv2.bitwise_and(
        edges,
        mask
    )

    # ---------------------------------------------------------
    # 5. Hough line detection
    # ---------------------------------------------------------

    lines = cv2.HoughLinesP(
        road_edges,
        rho=1,
        theta=np.pi / 180,
        threshold=40,
        minLineLength=40,
        maxLineGap=120
    )

    left_lines = []
    right_lines = []

    # ---------------------------------------------------------
    # IMPORTANT FIX
    # ---------------------------------------------------------

    if lines is not None:

        # Converts either:
        # (N,1,4)
        # or
        # (N,4)
        #
        # safely into:
        # (N,4)

        lines = np.asarray(
            lines
        ).reshape(-1, 4)

        for x1, y1, x2, y2 in lines:

            x1 = int(x1)
            y1 = int(y1)
            x2 = int(x2)
            y2 = int(y2)

            if x2 == x1:
                continue

            slope = (
                y2 - y1
            ) / (
                x2 - x1
            )

            # Ignore horizontal lines
            if abs(slope) < 0.35:
                continue

            if slope < 0:

                left_lines.append(
                    (x1, y1, x2, y2)
                )

            else:

                right_lines.append(
                    (x1, y1, x2, y2)
                )

    # ---------------------------------------------------------
    # 6. Draw detected lanes
    # ---------------------------------------------------------

    overlay = frame.copy()

    for x1, y1, x2, y2 in left_lines:

        cv2.line(
            overlay,
            (x1, y1),
            (x2, y2),
            (255, 0, 0),
            5
        )

    for x1, y1, x2, y2 in right_lines:

        cv2.line(
            overlay,
            (x1, y1),
            (x2, y2),
            (255, 0, 0),
            5
        )

    # ---------------------------------------------------------
    # 7. Estimate lane center
    # ---------------------------------------------------------

    image_center = w // 2

    lane_center = image_center

    if left_lines and right_lines:

        left_points = []

        for x1, y1, x2, y2 in left_lines:

            # choose point closer to bottom
            if y1 > y2:
                left_points.append(x1)
            else:
                left_points.append(x2)

        right_points = []

        for x1, y1, x2, y2 in right_lines:

            if y1 > y2:
                right_points.append(x1)
            else:
                right_points.append(x2)

        if left_points and right_points:

            left_position = int(
                np.median(left_points)
            )

            right_position = int(
                np.median(right_points)
            )

            lane_center = int(
                (
                    left_position
                    +
                    right_position
                ) / 2
            )

    # ---------------------------------------------------------
    # 8. Calculate steering correction
    # ---------------------------------------------------------

    error = (
        lane_center
        -
        image_center
    )

    steering = np.clip(
        error / max(w, 1) * 60,
        -15,
        15
    )

    steering = float(
        steering
    )

    # ---------------------------------------------------------
    # 9. Lane status
    # ---------------------------------------------------------

    if steering < -2:

        status = "STEER LEFT"

    elif steering > 2:

        status = "STEER RIGHT"

    else:

        status = "CENTERED"

    # ---------------------------------------------------------
    # 10. Draw planned path
    # ---------------------------------------------------------

    cv2.line(
        overlay,
        (image_center, h),
        (
            lane_center,
            int(h * 0.58)
        ),
        (0, 255, 255),
        5
    )

    # ---------------------------------------------------------
    # Display steering angle
    # ---------------------------------------------------------

    cv2.putText(
        overlay,
        f"Lane Steering: {steering:+.1f} deg",
        (
            int(w * 0.65),
            int(h * 0.90)
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (0, 255, 255),
        2
    )

    return (
        overlay,
        steering,
        status
    )

In [ ]:
# =====================================================================
# 11. ESTIMATE DISTANCE FROM BOUNDING BOX
# =====================================================================

def estimate_distance(

    box_height,
    frame_height

):

    """

    DEMONSTRATION ESTIMATE ONLY.

    Larger bounding box means object is probably closer.

    This is NOT calibrated metric depth.

    """

    if box_height <= 0:

        return 100.0


    normalized_height = (

        box_height
        /
        frame_height

    )


    # Simple nonlinear approximation

    distance = (

        4.5
        /
        max(
            normalized_height,
            0.03
        )

    )


    return float(

        np.clip(
            distance,
            5,
            120
        )

    )


# =====================================================================
# 12. APPROXIMATE TTC
# =====================================================================

def estimate_ttc(

    distance,

    object_key,

    box_height

):

    """

    Approximate TTC using change in apparent object size.

    If an object's bounding box gets larger,
    it is probably approaching.

    Demonstration only.

    """

    old_height = previous_box_heights.get(
        object_key
    )


    previous_box_heights[
        object_key
    ] = box_height


    if old_height is None:

        return float("inf")


    growth = (

        box_height
        -
        old_height

    )


    # Not getting visibly closer

    if growth <= 0.5:

        return float("inf")


    # Approximate TTC from optical expansion

    ttc = (

        box_height
        /
        max(
            growth * fps,
            0.001
        )

    )


    return float(

        np.clip(
            ttc,
            0.5,
            20
        )

    )


# =====================================================================
# 13. DETERMINISTIC FALLBACK AUTOPILOT
# =====================================================================

def fallback_decision(

    distance,
    ttc,
    lane_state

):


    # Critical risk

    if (

        distance < 12

        or

        (
            not math.isinf(ttc)
            and
            ttc < 2
        )

    ):

        return (

            "EMERGENCY_BRAKE",

            "Object critically close"

        )


    # Moderate risk

    if (

        distance < 25

        or

        (
            not math.isinf(ttc)
            and
            ttc < 4
        )

    ):

        return (

            "SLOW_DOWN",

            "Vehicle ahead is too close"

        )


    # Lane correction

    if lane_state == "STEER LEFT":

        return (

            "KEEP_LANE",

            "Correcting toward lane center"

        )


    if lane_state == "STEER RIGHT":

        return (

            "KEEP_LANE",

            "Correcting toward lane center"

        )


    return (

        "KEEP_LANE",

        "Road conditions acceptable"

    )


# =====================================================================
# 14. GEMINI HIGH-LEVEL DECISION
# =====================================================================

def ask_gemini(

    object_summary,
    distance,
    ttc,
    lane_state

):


    if not USE_GEMINI:

        return fallback_decision(

            distance,
            ttc,
            lane_state

        )


    if math.isinf(ttc):

        ttc_value = "infinite / no closing risk"

    else:

        ttc_value = f"{ttc:.1f} seconds"


    prompt = f"""

You are the HIGH-LEVEL tactical reasoning module
of an EDUCATIONAL Physical AI autonomous-driving demo.

You do NOT directly control the real vehicle.

Current perception:

Detected road objects:
{object_summary}

Nearest estimated object distance:
{distance:.1f} meters

Approximate TTC:
{ttc_value}

Lane status:
{lane_state}

Choose ONE high-level action:

KEEP_LANE
SLOW_DOWN
EMERGENCY_BRAKE

Safety rules:

- Safety has highest priority.
- Use EMERGENCY_BRAKE only for critical collision risk.
- Use SLOW_DOWN when the nearest vehicle is too close.
- Otherwise KEEP_LANE.
- Lane-centering steering is controlled separately.

Return exactly:

ACTION|short reason

"""


    try:

        response = (
            gemini_client.models.generate_content(

                model="gemini-2.5-flash",

                contents=prompt

            )
        )


        text = (
            response.text
            or ""
        ).strip()


        if "|" not in text:

            return fallback_decision(
                distance,
                ttc,
                lane_state
            )


        action, reason = text.split(
            "|",
            1
        )


        action = action.strip().upper()

        reason = reason.strip()


        if action not in [

            "KEEP_LANE",

            "SLOW_DOWN",

            "EMERGENCY_BRAKE"

        ]:

            return fallback_decision(
                distance,
                ttc,
                lane_state
            )


        return action, reason


    except Exception as e:

        return fallback_decision(

            distance,
            ttc,
            lane_state

        )


# =====================================================================
# 15. SAFETY CONTROLLER
# =====================================================================

def safety_controller(

    ai_action,
    distance,
    ttc

):


    # -------------------------------------------------------------
    # Critical safety override
    # -------------------------------------------------------------

    if (

        distance < 10

        or

        (
            not math.isinf(ttc)
            and
            ttc < 1.5
        )

    ):

        return (

            "EMERGENCY_BRAKE",

            "OVERRIDE - CRITICAL COLLISION RISK"

        )


    # -------------------------------------------------------------
    # Low TTC override
    # -------------------------------------------------------------

    if (

        (
            not math.isinf(ttc)
            and
            ttc < 3
        )

        and

        ai_action == "KEEP_LANE"

    ):

        return (

            "SLOW_DOWN",

            "OVERRIDE - LOW TTC"

        )


    return (

        ai_action,

        "APPROVED"

    )



In [ ]:

# =====================================================================
# 16. SIMULATED ACTUATOR CONTROLLER
# =====================================================================

def actuator_controller(

    action,
    lane_steering

):


    global ego_speed
    global target_speed
    global throttle
    global brake
    global steering_angle


    # Lane controller controls steering

    steering_angle = lane_steering


    if action == "KEEP_LANE":

        target_speed = 90

        throttle = 35

        brake = 0


    elif action == "SLOW_DOWN":

        target_speed = 55

        throttle = 0

        brake = 45


    elif action == "EMERGENCY_BRAKE":

        target_speed = 0

        throttle = 0

        brake = 100


    # -------------------------------------------------------------
    # Simple longitudinal dynamics
    # -------------------------------------------------------------

    if ego_speed < target_speed:

        ego_speed += 0.6


    elif ego_speed > target_speed:


        if brake >= 90:

            ego_speed -= 2.5

        elif brake > 0:

            ego_speed -= 1.0

        else:

            ego_speed -= 0.2


    ego_speed = float(

        np.clip(
            ego_speed,
            0,
            120
        )

    )


# =====================================================================
# 17. DRAW AUTONOMOUS DRIVING HUD
# =====================================================================

def draw_hud(

    frame,
    detected_count,
    nearest_object,
    distance,
    ttc,
    lane_state,
    ai_action,
    final_action,
    reason,
    safety

):


    h,w = frame.shape[:2]


    overlay = frame.copy()


    cv2.rectangle(

        overlay,

        (15,15),

        (min(650,w-15),330),

        (0,0,0),

        -1

    )


    cv2.addWeighted(

        overlay,

        0.58,

        frame,

        0.42,

        0,

        frame

    )


    if math.isinf(ttc):

        ttc_text = "INF"

    else:

        ttc_text = (
            f"{ttc:.1f} sec"
        )


    # -------------------------------------------------------------
    # Collision risk level
    # -------------------------------------------------------------

    if (

        distance < 10

        or

        (
            not math.isinf(ttc)
            and
            ttc < 1.5
        )

    ):

        risk = "CRITICAL"

        risk_color = (0,0,255)


    elif (

        distance < 25

        or

        (
            not math.isinf(ttc)
            and
            ttc < 4
        )

    ):

        risk = "HIGH"

        risk_color = (0,165,255)


    else:

        risk = "NORMAL"

        risk_color = (0,255,0)


    lines = [

        (
            "PHYSICAL AI - AUTONOMOUS DRIVING",
            (0,255,0),
            0.70
        ),

        (
            f"Speed: {ego_speed:.1f} km/h",
            (255,255,255),
            0.55
        ),

        (
            f"Objects detected: {detected_count}",
            (255,255,255),
            0.55
        ),

        (
            f"Nearest: {nearest_object}   "
            f"Estimated distance: {distance:.1f} m",
            (255,255,255),
            0.50
        ),

        (
            f"TTC: {ttc_text}   "
            f"Risk: {risk}",
            risk_color,
            0.55
        ),

        (
            f"Lane Control: {lane_state}",
            (0,255,255),
            0.52
        ),

        (
            f"Gemini Decision: {ai_action}",
            (0,255,255),
            0.52
        ),

        (
            f"Safety Layer: {safety}",
            (255,220,100),
            0.47
        ),

        (
            f"FINAL ACTION: {final_action}",
            risk_color,
            0.55
        ),

        (
            f"Steering: {steering_angle:+.1f} deg   "
            f"Throttle: {throttle:.0f}%   "
            f"Brake: {brake:.0f}%",
            (255,255,255),
            0.48
        )

    ]


    y = 45


    for text,color,size in lines:

        cv2.putText(

            frame,

            text[:100],

            (30,y),

            cv2.FONT_HERSHEY_SIMPLEX,

            size,

            color,

            2 if y == 45 else 1,

            cv2.LINE_AA

        )

        y += 29


    # -------------------------------------------------------------
    # Driver-seat steering wheel
    # -------------------------------------------------------------

    wheel_x = (
        w // 2
        +
        int(
            steering_angle * 3
        )
    )


    wheel_y = h - 15


    wheel_radius = max(
        45,
        int(h*0.10)
    )


    cv2.circle(

        frame,

        (wheel_x,wheel_y),

        wheel_radius,

        (40,40,40),

        15

    )


    cv2.line(

        frame,

        (wheel_x,wheel_y),

        (
            wheel_x,
            wheel_y-wheel_radius
        ),

        (40,40,40),

        12

    )


# =====================================================================
# 18. MAIN AUTONOMOUS DRIVING LOOP
# =====================================================================

frame_number = 0


# Gemini is not called every video frame.
# Tactical decision every ~2 seconds.

DECISION_INTERVAL = max(
    1,
    int(fps * 2)
)


print()
print("======================================================")
print("AUTONOMOUS DRIVING SIMULATION STARTED")
print("======================================================")
print()


while True:


    ret, frame = cap.read()


    if not ret:

        break


    if frame_number >= max_frames:

        break


    frame_number += 1


    # =================================================================
    # A. LANE PERCEPTION
    # =================================================================

    frame, lane_steering, lane_status = detect_lanes(
        frame
    )


    # =================================================================
    # B. YOLO OBJECT PERCEPTION
    # =================================================================

    results = model(

        frame,

        classes=ROAD_CLASSES,

        conf=0.35,

        verbose=False

    )


    result = results[0]


    detected_objects = []


    nearest_distance = float("inf")

    nearest_object = "None"

    nearest_ttc = float("inf")


    # =================================================================
    # C. PROCESS DETECTIONS
    # =================================================================

    for detection_index, box in enumerate(
        result.boxes
    ):


        x1,y1,x2,y2 = map(

            int,

            box.xyxy[0].tolist()

        )


        class_id = int(
            box.cls[0]
        )


        confidence = float(
            box.conf[0]
        )


        object_name = CLASS_NAMES.get(

            class_id,

            "Object"

        )


        box_height = (
            y2-y1
        )


        distance = estimate_distance(

            box_height,

            height

        )


        object_key = (
            object_name,
            detection_index
        )


        ttc = estimate_ttc(

            distance,

            object_key,

            box_height

        )


        detected_objects.append(

            {
                "type":
                    object_name,

                "distance":
                    round(
                        distance,
                        1
                    ),

                "confidence":
                    round(
                        confidence,
                        2
                    )
            }

        )


        # -------------------------------------------------------------
        # Find nearest object
        # -------------------------------------------------------------

        if distance < nearest_distance:


            nearest_distance = distance

            nearest_object = object_name

            nearest_ttc = ttc


        # -------------------------------------------------------------
        # Draw detection bounding box
        # -------------------------------------------------------------

        color = CLASS_COLORS.get(

            object_name,

            (0,255,0)

        )


        cv2.rectangle(

            frame,

            (x1,y1),

            (x2,y2),

            color,

            3

        )


        label = (

            f"{object_name} "

            f"{confidence:.2f} "

            f"~{distance:.0f}m"

        )


        cv2.putText(

            frame,

            label,

            (
                x1,
                max(
                    y1-10,
                    20
                )
            ),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.50,

            color,

            2,

            cv2.LINE_AA

        )


    # No objects detected

    if math.isinf(
        nearest_distance
    ):

        nearest_distance = 120.0


    # =================================================================
    # D. GEMINI TACTICAL DECISION
    # =================================================================

    if (

        frame_number == 1

        or

        frame_number
        %
        DECISION_INTERVAL
        ==
        0

    ):


        object_summary = json.dumps(

            detected_objects[:10]

        )


        gemini_action, gemini_reason = ask_gemini(

            object_summary,

            nearest_distance,

            nearest_ttc,

            lane_status

        )


        print(

            f"Time "
            f"{frame_number/fps:.1f}s | "

            f"Gemini: "
            f"{gemini_action} | "

            f"{gemini_reason}"

        )


    # =================================================================
    # E. SAFETY SUPERVISOR
    # =================================================================

    final_action, safety_status = safety_controller(

        gemini_action,

        nearest_distance,

        nearest_ttc

    )


    # =================================================================
    # F. ACTUATOR CONTROL
    # =================================================================

    actuator_controller(

        final_action,

        lane_steering

    )


    # =================================================================
    # G. DRAW HUD
    # =================================================================

    draw_hud(

        frame,

        len(
            detected_objects
        ),

        nearest_object,

        nearest_distance,

        nearest_ttc,

        lane_status,

        gemini_action,

        final_action,

        gemini_reason,

        safety_status

    )


    # =================================================================
    # H. WRITE OUTPUT FRAME
    # =================================================================

    writer.write(
        frame
    )


# =====================================================================
# 19. CLEANUP
# =====================================================================

cap.release()

writer.release()


print()
print("Video processing completed.")


# =====================================================================
# 20. CONVERT TO H.264 FOR GOOGLE COLAB
# =====================================================================

os.system(

    f'ffmpeg -y '
    f'-loglevel error '
    f'-i "{raw_output}" '
    f'-vcodec libx264 '
    f'-pix_fmt yuv420p '
    f'"{final_output}"'

)


# =====================================================================
# 21. DISPLAY AUTONOMOUS DRIVING VIDEO
# =====================================================================

with open(

    final_output,

    "rb"

) as f:

    video_bytes = f.read()


encoded = b64encode(
    video_bytes
).decode()


print()
print("======================================================")
print("PHYSICAL AI AUTONOMOUS DRIVING DEMO")
print("======================================================")


display(

    HTML(

        f'''

        <video
            width="1000"
            controls
            autoplay
            loop>

            <source
                src="data:video/mp4;base64,{encoded}"
                type="video/mp4">

        </video>

        '''

    )

)


# =====================================================================
# 22. DOWNLOAD RESULT
# =====================================================================

print()
print(
    "Output video:",
    final_output
)

files.download(
    final_output
)